# UHPC

## Data Loading & Feature Isolation


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 50-feature dataset

df_drop = pd.read_csv(".//initial_data_preparation/Datasets/processed/uhpc_dataset/semantic_recoding_features_50.csv")
#df_drop = pd.read_csv(".//initial_data_preparation/Datasets/processed/uhpc_dataset/semantic_recoding_features_20.csv")

# Apply drops 
df_drop= df_drop.drop(columns=['cement_grade'])
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_drop[fiber_cols] = df_drop[fiber_cols].fillna(0) 

groups = df_drop['reference']

# Proceed to split data
Y = df_drop['cs_28d'] 
X = df_drop.drop(columns=['cs_28d', 'reference'])


In [3]:
# Identify all categorical columns
str_cols = X.select_dtypes(include=['object', 'str']).columns

# Sort text columns into One-Hot (<= 10 categories) and Target Encoding (> 10 categories)
one_hot_encode_cols = str_cols[X[str_cols].nunique() <= 10].tolist()
target_encode_cols = str_cols[X[str_cols].nunique() > 10].tolist()

# Identify all numeric columns
numeric_features = X.select_dtypes(include='number').columns.tolist()

# Optional: Print them out to verify
print(f"One-Hot columns: {len(one_hot_encode_cols)}")
print(f"Target Encoded columns: {len(target_encode_cols)}")
print(f"Numeric columns: {len(numeric_features)}")

One-Hot columns: 5
Target Encoded columns: 4
Numeric columns: 24


In [4]:
df_drop.shape

(2072, 35)

## Leave one group out

In [5]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from pipeline_w8 import preprocessing_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
import numpy as np

In [ ]:
# Apply Leave One Group Out cv
logo = LeaveOneGroupOut()

results = []

models = [XGBRegressor(random_state=42), HistGradientBoostingRegressor(random_state=42), AdaBoostRegressor(random_state=42)]

# Create empty lists to hold all Y_test and Y_pred
Y_test_all = []
Y_pred_all = []

results_all = []
# Dictionary to hold Y_test and Y_pred for each model separately 
dict_all = {model.__class__.__name__: {'Y_test': [], 'Y_pred': []} for model in models}

# Initialize loop to leave one group out
for train_index, test_index in logo.split(X, Y, groups=groups):

    # Isolate the data 
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]
    
    # Save the publication name 
    publ = groups.iloc[test_index].values[0]

    # Pipeline loop for 3 models
    for model in models:
        # Get the name of the current model
        model_name = model.__class__.__name__

        # Initialize pipeline
        pipe = preprocessing_pipeline(one_hot_encode_cols,target_encode_cols,numeric_features,estimator=model)
        
        pipe.fit(X_train, Y_train)
        Y_pred = pipe.predict(X_test)
        
        dict_all[model_name]['Y_test'].extend(Y_test)
        dict_all[model_name]['Y_pred'].extend(Y_pred)

        mae = mean_absolute_error(Y_test,Y_pred)
        rmse = root_mean_squared_error(Y_test,Y_pred)
        n_rows = len(Y_test)

        if n_rows > 1:
            r2 = r2_score(Y_test, Y_pred)
        else:
            r2 = None

        mean_residual = np.mean(Y_test - Y_pred)

        results.append({
            'Experiment': publ,
            'Model': model.__class__.__name__,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'Num. rows': n_rows,
            'Mean Residual': mean_residual
        })

for model_name, data in dict_all.items():
    mae_all = mean_absolute_error(data['Y_test'], data['Y_pred'])
    rmse_all = root_mean_squared_error(data['Y_test'], data['Y_pred'])
    r2_all = r2_score(data['Y_test'], data['Y_pred'])
    
    results_all.append({
        'Model':model_name,
        'MAE':mae_all,
        'RMSE':rmse_all,
        'r2_all':r2_all
    })

df_results = pd.DataFrame(results)
df_results_all = pd.DataFrame(results_all)


In [7]:
df_results.sort_values(by=['MAE'], ascending=False)

,Experiment,Model,MAE,RMSE,R2,Num. rows,Mean Residual
173,Ref-152-Research,AdaBoostRegressor,65.124332,65.137306,-2509.573129,2,65.124332
96,Ref-129-Research,XGBRegressor,61.128876,61.128876,NaN,1,-61.128876
98,Ref-129-Research,AdaBoostRegressor,58.221572,58.221572,NaN,1,-58.221572
202,Ref-161-Research,HistGradientBoostingRegressor,56.600470,57.749204,-21.614229,6,-56.600470
119,Ref-136-Research,AdaBoostRegressor,53.644701,58.037403,-2.663206,9,53.644701
...,...,...,...,...,...,...,...
342,Ref-52-Research,XGBRegressor,1.806244,2.500912,-0.077340,6,0.809891
169,Ref-151-Research,HistGradientBoostingRegressor,1.749192,1.753517,0.972110,2,-1.749192
487,Ref-97-Research,HistGradientBoostingRegressor,1.300382,1.300382,NaN,1,1.300382
65,Ref-119-Research,AdaBoostRegressor,1.282330,1.282330,NaN,1,1.282330


In [8]:
df_results_all

,Model,MAE,RMSE,r2_all
0,XGBRegressor,18.438446,23.979130,0.565162
1,HistGradientBoostingRegressor,18.733623,24.392696,0.550033
2,AdaBoostRegressor,20.413099,25.493699,0.508497


## Raw Results & Residual Analysis

In [9]:
raw_avg = df_results.groupby('Model')[['MAE', 'RMSE', 'R2']].mean()
raw_avg

,MAE,RMSE,R2
Model,,,
AdaBoostRegressor,20.575094,22.833911,-30.274516
HistGradientBoostingRegressor,18.053035,20.427740,-15.580219
XGBRegressor,17.957445,20.447916,-12.515081


## Stable Performance

In [10]:
conditions = (df_results['MAE'] >= 20) & (df_results['Num. rows'] <= 5)

# Excluding outliers results
df_stable_results = df_results[~conditions] 
st_avg = df_stable_results.groupby('Model')[['MAE', 'RMSE', 'R2']].mean()
st_avg

,MAE,RMSE,R2
Model,,,
AdaBoostRegressor,18.065653,20.519834,-4.799889
HistGradientBoostingRegressor,16.407318,18.950215,-5.297420
XGBRegressor,16.243203,18.860967,-4.568138


In [11]:
# Isolate  worst publication
worst_pub = df_drop[df_drop['reference'] == 'Ref-136-Research']

# Display the numeric features
worst_pub.select_dtypes(include='number').T

,1571,1572,1573,1574,1575,1576,1577,1578,1579
cement,833.330000,956.180000,948.620000,941.180000,851.060000,991.740000,975.610000,991.740000,1061.950000
silica_fume,208.330000,239.040000,237.150000,235.290000,212.770000,247.930000,243.900000,247.930000,265.490000
fly_ash,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
limestone_powder,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
quartz_powder,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
glass_powder,208.330000,239.040000,237.150000,235.290000,212.770000,247.930000,243.900000,247.930000,265.490000
rice_husk_ash,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
metakaolin,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ggbfs,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
slag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [12]:
worst_pub.describe()

,cement,silica_fume,fly_ash,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,ggbfs,slag,...,sand,sand_max_size,fiber1_amount,fiber1_length,fiber1_diameter,fiber2_amount,water,sp_amount,curing_temp,cs_28d
count,9.000000,9.000000,9.0,9.0,9.0,9.000000,9.0,9.0,9.0,9.0,...,9.000000,9.0,9.000000,9.000000,9.000000,9.0,9.000000,9.000000,9.0,9.000000
mean,950.156667,237.536667,0.0,0.0,0.0,237.536667,0.0,0.0,0.0,0.0,...,974.770000,0.8,138.666667,6.666667,0.122222,0.0,189.624788,9.313779,20.0,235.222222
std,70.880014,17.719332,0.0,0.0,0.0,17.719332,0.0,0.0,0.0,0.0,...,106.314645,0.0,146.069333,6.519202,0.120185,0.0,14.557768,2.724265,0.0,32.162780
min,833.330000,208.330000,0.0,0.0,0.0,208.330000,0.0,0.0,0.0,0.0,...,807.080000,0.8,0.000000,0.000000,0.000000,0.0,169.411765,4.500000,20.0,194.000000
25%,941.180000,235.290000,0.0,0.0,0.0,235.290000,0.0,0.0,0.0,0.0,...,912.400000,0.8,0.000000,0.000000,0.000000,0.0,180.425532,10.245059,20.0,213.000000
50%,956.180000,239.040000,0.0,0.0,0.0,239.040000,0.0,0.0,0.0,0.0,...,965.740000,0.8,156.000000,8.000000,0.200000,0.0,186.454183,10.536585,20.0,230.000000
75%,991.740000,247.930000,0.0,0.0,0.0,247.930000,0.0,0.0,0.0,0.0,...,988.240000,0.8,234.000000,13.000000,0.200000,0.0,198.347107,10.710744,20.0,261.000000
max,1061.950000,265.490000,0.0,0.0,0.0,265.490000,0.0,0.0,0.0,0.0,...,1150.000000,0.8,390.000000,13.000000,0.300000,0.0,219.823009,11.469027,20.0,291.000000


The original dataset has 0 glass powder in at least 75% of its rows. The glass powder amount causes this massive difference in MAE. 

In [16]:
# Isolate  worst publication
worst_pub_2 = df_drop[df_drop['reference'] == 'Ref-161-Research']

# Display the numeric features
worst_pub_2.select_dtypes(include='number').T

,2008,2009,2010,2011,2012,2013
cement,1079.00,1025.00,971.00,917.00,863.00,809.00
silica_fume,0.00,54.00,108.00,162.00,216.00,270.00
fly_ash,0.00,0.00,0.00,0.00,0.00,0.00
limestone_powder,0.00,0.00,0.00,0.00,0.00,0.00
quartz_powder,0.00,0.00,0.00,0.00,0.00,0.00
glass_powder,0.00,0.00,0.00,0.00,0.00,0.00
rice_husk_ash,0.00,0.00,0.00,0.00,0.00,0.00
metakaolin,0.00,0.00,0.00,0.00,0.00,0.00
ggbfs,0.00,0.00,0.00,0.00,0.00,0.00
slag,0.00,0.00,0.00,0.00,0.00,0.00


In [17]:
worst_pub_2.describe()

,cement,silica_fume,fly_ash,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,ggbfs,slag,...,sand,sand_max_size,fiber1_amount,fiber1_length,fiber1_diameter,fiber2_amount,water,sp_amount,curing_temp,cs_28d
count,6.000000,6.000000,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,...,6.0,6.00,6.0,6.0,6.000000e+00,6.0,6.000000,6.000000,6.0,6.000000
mean,944.000000,135.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,149.166667,11.133333,20.0,106.833333
std,101.024749,101.024749,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.0,3.040471e-17,0.0,13.629625,3.478889,0.0,13.302882
min,809.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,125.000000,8.600000,20.0,83.000000
25%,876.500000,67.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,144.250000,8.600000,20.0,102.500000
50%,944.000000,135.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,155.000000,9.700000,20.0,112.000000
75%,1011.500000,202.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,159.000000,12.375000,20.0,115.500000
max,1079.000000,270.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,913.0,4.75,156.0,13.0,2.000000e-01,0.0,159.000000,17.300000,20.0,118.000000


This example shows low variance and huge sand ingredient amount and size cause this worst result.

In [13]:
df_drop.describe()

,cement,silica_fume,fly_ash,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,ggbfs,slag,...,sand,sand_max_size,fiber1_amount,fiber1_length,fiber1_diameter,fiber2_amount,water,sp_amount,curing_temp,cs_28d
count,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,...,2072.000000,1796.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2064.000000,1915.000000,2072.000000
mean,786.164049,180.836110,59.545347,13.614966,58.928427,32.377780,2.341776,4.903378,42.046187,17.086573,...,897.897741,1.366793,93.681735,9.488755,0.122138,2.547403,195.271364,31.865909,56.187833,150.200085
std,202.424895,90.045347,143.872803,62.890751,113.392827,123.678095,22.752543,31.737475,141.964661,97.282349,...,296.780711,1.155117,90.487545,8.859023,0.106274,14.264307,43.374299,17.624763,47.580591,36.372613
min,170.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.050000,0.000000,0.000000,0.000000,0.000000,110.000000,2.640000,20.000000,80.000000
25%,664.000000,133.425000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,815.000000,0.600000,0.000000,0.000000,0.000000,0.000000,166.700000,17.470000,20.000000,124.812500
50%,788.500000,197.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,960.000000,0.630000,80.950000,12.700000,0.150000,0.000000,184.000000,30.000000,23.000000,145.200000
75%,900.000000,230.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1056.000000,2.000000,156.000000,13.000000,0.200000,0.000000,214.400000,44.200000,90.000000,170.000000
max,1856.700000,617.650000,1152.000000,1058.200000,528.000000,1067.000000,481.060000,510.000000,768.000000,1100.000000,...,1994.000000,4.750000,624.000000,60.000000,0.550000,195.000000,355.150000,151.700000,210.000000,298.000000


In [14]:
from IPython.display import display, Markdown

display(Markdown("### 1. Raw Performance (All Data)"))
display(raw_avg)

display(Markdown("### 2. Stable Performance (Outliers Removed)"))
display(st_avg)

display(Markdown("### 3. Global result"))
display(df_results_all)

### 1. Raw Performance (All Data)

,MAE,RMSE,R2
Model,,,
AdaBoostRegressor,20.575094,22.833911,-30.274516
HistGradientBoostingRegressor,18.053035,20.427740,-15.580219
XGBRegressor,17.957445,20.447916,-12.515081


### 2. Stable Performance (Outliers Removed)

,MAE,RMSE,R2
Model,,,
AdaBoostRegressor,18.065653,20.519834,-4.799889
HistGradientBoostingRegressor,16.407318,18.950215,-5.297420
XGBRegressor,16.243203,18.860967,-4.568138


### 3. Global result

,Model,MAE,RMSE,r2_all
0,XGBRegressor,18.438446,23.979130,0.565162
1,HistGradientBoostingRegressor,18.733623,24.392696,0.550033
2,AdaBoostRegressor,20.413099,25.493699,0.508497


In [20]:
# Count the total number of unique publications
num_publications = groups.nunique()
print(f"Total number of unique publications: {num_publications}")

# group sizes
group_sizes = groups.value_counts()

# Print a quick statistical summary of the group sizes
print("\nGroup Size Summary:")
print(group_sizes.describe())

# The top 5 largest and top 5 smallest publications
print("\nLargest 5 Publications:")
print(group_sizes.head(10))
print("\nSmallest 5 Publications:")
print(group_sizes.tail(5))

Total number of unique publications: 165

Group Size Summary:
count    165.000000
mean      12.557576
std       15.085266
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      112.000000
Name: count, dtype: float64

Largest 5 Publications:
reference
Ref-144-Research    112
Ref-121-Research     80
Ref-141-Research     73
Ref-48-Research      72
Ref-85-Research      64
Ref-139-Research     51
Ref-135-Research     43
Ref-153-Research     43
Ref-116-Research     36
Ref-65-Research      30
Name: count, dtype: int64

Smallest 5 Publications:
reference
Ref-152-Research    2
Ref-35-Research     1
Ref-97-Research     1
Ref-119-Research    1
Ref-129-Research    1
Name: count, dtype: int64
